# Day 14: Complex Data Structures, Comprehensions & Contact Management System (CMS)
**Xeven Solutions — AI Engineer Internship Program**  
**Track:** NLP & LangChain Specialization  

---

### Project Architecture Overview:
This notebook demonstrates the complete implementation of a modular, in-memory **Contact Management System (CMS)** featuring:
1. **Data Structure:** Nested dictionary of dictionaries: `{id: {name, phone, email, tags: set, notes: list}}`
2. **CRUD Operations:** `add_contact()`, `search_contacts()`, `update_contact()`, `delete_contact()`
3. **Advanced Search:** Multi-criteria filtering using dictionary & list comprehensions
4. **Tag Management:** Set operations (`.add()`, `.discard()`, set intersections)
5. **File Persistence:** JSON save/load pipeline with Set $\leftrightarrow$ List conversions & exception handling
6. **Analytics:** Summary statistics & tag frequency counts using `collections.Counter`

## 1. Initial Data Structure Setup
We use an outer dictionary keyed by unique integer IDs ($O(1)$ lookup), with `tags` as a Python `set` (ensuring uniqueness) and `notes` as a `list` (chronological interaction logs).

In [1]:
import json
import os
from collections import Counter

# In-memory relational data structure
contacts = {
    1: {
        "name": "Awais Ramzan",
        "phone": "+923011939995",
        "email": "awais@gmail.com",
        "tags": {"Python", "AI", "ML"},
        "notes": ["AI Engineer", "Met at hackathon"]
    },
    2: {
        "name": "Ahmad Ali",
        "phone": "+923059839320",
        "email": "ahmad@gmail.com",
        "tags": {"Python", "Web"},
        "notes": ["Frontend Developer"]
    },
    3: {
        "name": "Ali Hassan",
        "phone": "+923001234567",
        "email": "ali@gmail.com",
        "tags": {"Python", "Django"},
        "notes": ["Backend Developer"]
    }
}

print(f"Initial contacts loaded: {len(contacts)}")

Initial contacts loaded: 3


## 2. Core CRUD Operations
Functions for Adding (with duplicate prevention), Searching, Updating (preserving untouched fields), and Deleting contacts.

In [2]:
def add_contact(contacts, name, phone, email, tags=None, notes=None):
    """Adds a new contact with unique ID and duplicate phone check."""
    # Duplicate phone validation
    for cid, c in contacts.items():
        if c["phone"] == phone:
            return (False, f"Contact already exists with phone '{phone}' (ID: {cid})", None)

    new_id = max(contacts.keys(), default=0) + 1
    new_contact = {
        "name": name.strip(),
        "phone": phone.strip(),
        "email": email.strip(),
        "tags": set(tags) if tags else set(),
        "notes": list(notes) if notes else []
    }
    contacts[new_id] = new_contact
    return (True, f"Contact '{name}' added with ID {new_id}!", new_contact)


def search_contacts(contacts, query):
    """Searches contacts by exact or partial match on name, phone, or email."""
    q = query.strip().lower()
    matches = {
        cid: c for cid, c in contacts.items()
        if q in c["name"].lower() or q in c["phone"].lower() or q in c["email"].lower()
    }
    if matches:
        return (True, f"Found {len(matches)} match(es):", matches)
    return (False, "No matching contacts found.", {})


def update_contact(contacts, contact_id, name=None, phone=None, email=None, new_note=None):
    """Updates contact details by ID."""
    if contact_id not in contacts:
        return (False, f"Contact ID {contact_id} not found!")

    c = contacts[contact_id]
    if name:
        c["name"] = name.strip()
    if phone:
        c["phone"] = phone.strip()
    if email:
        c["email"] = email.strip()
    if new_note:
        c["notes"].append(new_note.strip())

    return (True, f"Contact ID {contact_id} updated successfully!")


def delete_contact(contacts, contact_id):
    """Deletes a contact by ID."""
    if contact_id in contacts:
        deleted = contacts.pop(contact_id)
        return (True, f"Deleted contact '{deleted['name']}' (ID: {contact_id})")
    return (False, f"Contact ID {contact_id} not found!")

## 3. Testing CRUD Operations

In [3]:
# 1. Add Sara Khan
res_add = add_contact(contacts, "Sara Khan", "+923041112233", "sara@gmail.com", tags={"Python", "Data Science"}, notes=["Joined via LinkedIn"])
print("Add Contact:", res_add)

# 2. Search for Awais
res_search = search_contacts(contacts, "Awais")
print("Search 'Awais':", res_search)

# 3. Update Awais (ID 1)
res_update = update_contact(contacts, 1, new_note="Promoted to Senior AI Engineer")
print("Update ID 1:", res_update)

# 4. Delete Ali Hassan (ID 3)
res_delete = delete_contact(contacts, 3)
print("Delete ID 3:", res_delete)

print("Total remaining:", len(contacts))

Add Contact: (True, "Contact 'Sara Khan' added with ID 4!", {'name': 'Sara Khan', 'phone': '+923041112233', 'email': 'sara@gmail.com', 'tags': {'Data Science', 'Python'}, 'notes': ['Joined via LinkedIn']})
Search 'Awais': (True, 'Found 1 match(es):', {1: {'name': 'Awais Ramzan', 'phone': '+923011939995', 'email': 'awais@gmail.com', 'tags': {'AI', 'Python', 'ML'}, 'notes': ['AI Engineer', 'Met at hackathon']}})
Update ID 1: (True, 'Contact ID 1 updated successfully!')
Delete ID 3: (True, "Deleted contact 'Ali Hassan' (ID: 3)")
Total remaining: 3


## 4. Advanced Search with Comprehensions
Using dictionary comprehensions to perform compound filtering on `name`, `tag`, and `keyword` (across notes and email) with case-insensitive matching.

In [4]:
def advanced_search(contacts, name=None, tag=None, keyword=None):
    """Multi-criteria search using dictionary comprehensions."""
    results = {
        cid: c for cid, c in contacts.items()
        if (
            (name is None or name.lower() in c["name"].lower())
            and
            (tag is None or any(tag.lower() == t.lower() for t in c["tags"]))
            and
            (keyword is None or (
                keyword.lower() in c["name"].lower() or
                keyword.lower() in c["email"].lower() or
                any(keyword.lower() in note.lower() for note in c["notes"])
            ))
        )
    }
    return results

# Search by tag
python_matches = advanced_search(contacts, tag="Python")
print("Contacts with 'Python' tag:", list(python_matches.keys()))

# Search by keyword
kw_matches = advanced_search(contacts, keyword="hackathon")
print("Contacts with keyword 'hackathon':", list(kw_matches.keys()))

Contacts with 'Python' tag: [1, 2, 4]
Contacts with keyword 'hackathon': [1]


## 5. Tag Management Using Set Operations
Demonstrating set algebra: `.add()`, `.discard()`, and set membership to maintain zero-duplicate categories.

In [5]:
def add_tag(contacts, contact_id, tag_name):
    """Adds a tag to a contact using set.add()."""
    if contact_id not in contacts:
        return (False, f"Contact ID {contact_id} not found!")
    contacts[contact_id]["tags"].add(tag_name.strip())
    return (True, f"Tag '{tag_name}' added to contact ID {contact_id}.")


def remove_tag(contacts, contact_id, tag_name):
    """Removes a tag from a contact using set.discard()."""
    if contact_id not in contacts:
        return (False, f"Contact ID {contact_id} not found!")
    if tag_name not in contacts[contact_id]["tags"]:
        return (False, f"Tag '{tag_name}' not found on contact ID {contact_id}.")
    contacts[contact_id]["tags"].discard(tag_name)
    return (True, f"Tag '{tag_name}' removed from contact ID {contact_id}.")


def find_by_tag(contacts, tag_name):
    """Finds contacts that have a specific tag using set membership."""
    tag_clean = tag_name.strip().lower()
    matches = {
        cid: c for cid, c in contacts.items()
        if any(t.lower() == tag_clean for t in c["tags"])
    }
    return matches

# Test tag operations
print("Add Tag:", add_tag(contacts, 1, "FastAPI"))
print("Tags for ID 1:", contacts[1]["tags"])
print("Find by tag 'Web':", list(find_by_tag(contacts, "Web").keys()))

Add Tag: (True, "Tag 'FastAPI' added to contact ID 1.")
Tags for ID 1: {'AI', 'Python', 'ML', 'FastAPI'}
Find by tag 'Web': [2]


## 6. JSON File Persistence (Save & Load with Error Handling)
Handles the non-serializability of `set` by converting `set` $\to$ `list` upon saving and `list` $\to$ `set` upon loading.

In [6]:
def save_to_json(contacts, filename="contacts_demo.json"):
    """Saves contacts converting Sets to Lists."""
    try:
        serializable = {
            str(cid): {
                "name": c["name"],
                "phone": c["phone"],
                "email": c["email"],
                "tags": list(c["tags"]),
                "notes": c["notes"]
            }
            for cid, c in contacts.items()
        }
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(serializable, f, indent=4)
        return (True, f"Saved {len(contacts)} contacts to '{filename}' successfully!")
    except Exception as e:
        return (False, f"Failed to save: {str(e)}")


def load_from_json(filename="contacts_demo.json"):
    """Loads contacts converting Lists back to Sets with error handling."""
    if not os.path.exists(filename):
        return (False, f"File '{filename}' does not exist.", {})

    try:
        with open(filename, "r", encoding="utf-8") as f:
            raw = json.load(f)
        loaded = {}
        for cid_str, c in raw.items():
            loaded[int(cid_str)] = {
                "name": c.get("name", ""),
                "phone": c.get("phone", ""),
                "email": c.get("email", ""),
                "tags": set(c.get("tags", [])),
                "notes": list(c.get("notes", []))
            }
        return (True, f"Loaded {len(loaded)} contacts from '{filename}'!", loaded)
    except json.JSONDecodeError:
        return (False, f"Error: '{filename}' contains invalid JSON formatting.", {})
    except Exception as e:
        return (False, f"Unexpected error: {str(e)}", {})

# Test persistence
print("Save Status:", save_to_json(contacts, "contacts_demo.json"))
print("Load Status:", load_from_json("contacts_demo.json"))

# Clean up test demo file
if os.path.exists("contacts_demo.json"):
    os.remove("contacts_demo.json")

Save Status: (True, "Saved 3 contacts to 'contacts_demo.json' successfully!")
Load Status: (True, "Loaded 3 contacts from 'contacts_demo.json'!", {1: {'name': 'Awais Ramzan', 'phone': '+923011939995', 'email': 'awais@gmail.com', 'tags': {'AI', 'Python', 'ML', 'FastAPI'}, 'notes': ['AI Engineer', 'Met at hackathon', 'Promoted to Senior AI Engineer']}, 2: {'name': 'Ahmad Ali', 'phone': '+923059839320', 'email': 'ahmad@gmail.com', 'tags': {'Python', 'Web'}, 'notes': ['Frontend Developer']}, 4: {'name': 'Sara Khan', 'phone': '+923041112233', 'email': 'sara@gmail.com', 'tags': {'Data Science', 'Python'}, 'notes': ['Joined via LinkedIn']}})


## 7. Statistics & Analytics
Computing system metrics using `collections.Counter` to track tag distribution and top categories.

In [7]:
def get_statistics(contacts):
    """Calculates total contacts, tag usage distribution, and top tags."""
    total = len(contacts)
    all_tags = []
    for c in contacts.values():
        all_tags.extend(list(c["tags"]))

    tag_counter = Counter(all_tags)
    return {
        "total_contacts": total,
        "total_unique_tags": len(tag_counter),
        "most_used_tags": tag_counter.most_common(3),
        "tag_counts": dict(tag_counter)
    }

stats = get_statistics(contacts)
print("=" * 45)
print("              SYSTEM STATISTICS              ")
print("=" * 45)
print(f"  Total Contacts:     {stats['total_contacts']}")
print(f"  Unique Tags:        {stats['total_unique_tags']}")
print("\n  Top Most Used Tags:")
for tag, count in stats['most_used_tags']:
    print(f"    - {tag}: {count} contact(s)")
print("\n  Tag Distribution:")
print("   ", stats['tag_counts'])
print("=" * 45)

              SYSTEM STATISTICS              
  Total Contacts:     3
  Unique Tags:        5

  Top Most Used Tags:
    - Python: 3 contact(s)
    - AI: 1 contact(s)
    - ML: 1 contact(s)

  Tag Distribution:
    {'AI': 1, 'Python': 3, 'ML': 1, 'FastAPI': 1, 'Web': 1, 'Data Science': 1}
